# Estymacja systemu popytu na energię w gospodarstwach domowych metodą pozornie niezwiązanych regresji

## Podsumowanie zarządcze

Regionalne przedsiębiorstwo użyteczności publicznej estymuje dwurównaniowy **system popytu na energię** w gospodarstwach domowych — udziały budżetowe energii elektrycznej i gazu ziemnego jako funkcje ceny własnej, ceny krzyżowej oraz dochodu gospodarstwa — za pomocą **PROC SYSLIN** metodą pozornie niezwiązanych regresji (SUR). Dwa równania udziałów dzielą wspólny budżet gospodarstwa, więc ich błędy są skorelowane krzyżowo; SUR estymuje równania łącznie, aby wykorzystać tę korelację, odzyskując spójne znakowo efekty ceny własnej i krzyżowej oraz dostarczając kowariancji międzyrównaniowej i testów restrykcji, których potrzebuje analityk popytu.

## Źródła danych

| Zbiór danych | Wiersze | Ziarno | Zmienne kluczowe | Opis |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | jeden wiersz na miesięczną obserwację rynku | `month`, `lp_elec`, `lp_gas`, `lincome`, `w_elec`, `w_gas` | Syntetyczny miesięczny panel rynku energii dla gospodarstw domowych. `lp_elec`/`lp_gas` to logarytmy realnych cen energii elektrycznej i gazu ziemnego; `lincome` to logarytm realnego dochodu gospodarstwa; `w_elec`/`w_gas` to udziały budżetowe wydatków na energię elektryczną i gaz ziemny, wygenerowane ze znanej struktury popytu w stylu AIDS plus skorelowany szum międzyrównaniowy. |

# Estymacja systemu popytu na energię w gospodarstwach domowych metodą pozornie niezwiązanych regresji

Regionalne przedsiębiorstwo gazowo-elektryczne chce zrozumieć, jak gospodarstwa domowe przenoszą swój budżet energetyczny między **energię elektryczną** a **gaz ziemny** wraz ze zmianą cen względnych i dochodu. Naturalnym ujęciem jest **system popytu**: zestaw równań udziałów budżetowych estymowanych łącznie.

Dwie cechy sprawiają, że estymacja łączna jest właściwym narzędziem:

- Równania udziałów czerpią ze wspólnego budżetu gospodarstwa, więc ich błędy są **skorelowane krzyżowo** — gdy gospodarstwo wydaje więcej na energię elektryczną, wydaje mniej na gaz. Estymacja równań razem metodą **pozornie niezwiązanych regresji (SUR)** wykorzystuje tę korelację dla efektywności.
- Teoria ekonomiczna narzuca **restrykcje liniowe** (sumowanie do jedności, jednorodność, symetria), które estymator systemowy może egzekwować bezpośrednio.

`PROC SYSLIN` z opcją `SUR` jest stworzony dokładnie do tej sytuacji. Dopasowuje każde równanie udziału, estymuje międzyrównaniową kowariancję błędów i ponownie dopasowuje system z tą kowariancją, dostarczając efektywne, spójne z teorią elastyczności — wraz z macierzą kowariancji międzymodelowej i łącznymi testami restrykcji.

W tym notatniku:

1. Generujemy syntetyczny miesięczny panel rynku energii ze znanej struktury udziałów.
2. Dopasowujemy dwurównaniowy system udziałów metodą SUR.
3. Porównujemy łączne dopasowanie SUR z regresją OLS równanie po równaniu.
4. Nakładamy restrykcję jednorodności i odczytujemy jej łączny test F.
5. Wyodrębniamy tabelę współczynników do raportowania elastyczności.

## Krok 1 — Generowanie syntetycznego panelu rynku energii

Symulujemy 60 miesięcznych obserwacji małego **dwutowarowego systemu popytu na energię** (energia elektryczna i gaz ziemny). Proces generujący dane to zlinearyzowany model udziałów budżetowych w stylu AIDS:

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

Celowo wbudowujemy **korelację międzyrównaniową**: gdy gospodarstwa wydają więcej na energię elektryczną, wydają mniej na gaz, więc `e1` i `e2` są ujemnie skorelowane. Wspólny czynnik cenowy rynku energii również napędza oba logarytmy cen razem. To właśnie te cechy wynagradzają łączną estymację SUR w porównaniu z dopasowaniem każdego równania osobno.

In [1]:
DANE energy;
   CALL streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   POWTÓRZ month = 1 TO 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      WYJŚCIE;
   KONIEC;

   ZACHOWAJ month lp_elec lp_gas lincome w_elec w_gas;
WYKONAJ;

PROCEDURA ŚREDNIE DANE=energy n mean std MIN MAX maxdec=3;
   ZMIENNA w_elec w_gas lp_elec lp_gas lincome;
WYKONAJ;

                                                  The MEANS Procedure

 Variable        N           Mean     Std Dev     Minimum     Maximum
 --------------------------------------------------------------------
 w_elec         60          0.440       0.011       0.417       0.464
 w_gas          60          0.228       0.014       0.198       0.256
 lp_elec        60          4.617       0.142       4.354       4.911
 lp_gas         60          4.211       0.162       3.818       4.566
 lincome        60         10.916       0.088      10.723      11.174
 --------------------------------------------------------------------




NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Krok 2 — Bazowa estymacja SUR systemu popytu

Estymujemy oba równania udziałów łącznie. Opcja `SUR` mówi `PROC SYSLIN`, aby oszacował międzyrównaniową kowariancję błędów i użył jej do efektywnego dopasowania feasible-GLS. Dwie oznaczone instrukcje `MODEL` (`elec` i `gas`) definiują system; każda regresuje udział budżetowy na dwóch logarytmach cen i logarytmie dochodu.

SYSLIN raportuje estymaty parametrów każdego równania oraz, na końcu, **macierz kowariancji międzymodelowej (Cross-Model Covariance Matrix)** — oszacowaną kowariancję błędów obu równań, którą wykorzystuje SUR.

In [2]:
PROCEDURA syslin DANE=energy ON;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
WYKONAJ;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: w_elec

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: w_gas

  Number of Observations                       60
  SSE                                      0.


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Krok 3 — Porównanie z OLS równanie po równaniu

Aby zobaczyć, co daje nam SUR, ponownie dopasowujemy te same dwa równania zwykłą metodą najmniejszych kwadratów (metoda domyślna, jedno równanie naraz). OLS ignoruje międzyrównaniową korelację błędów, więc jest zgodny, ale mniej efektywny niż SUR, gdy ta korelacja jest niezerowa — jak jest tutaj z konstrukcji.

Porównanie obu tabel współczynników pokazuje, jak estymaty i ich błędy standardowe przesuwają się, gdy uwzględni się strukturę systemu.

In [3]:
PROCEDURA syslin DANE=energy ols;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
WYKONAJ;


                       The SYSLIN Procedure

                  OLS Estimation

  Model elec Dependent Variable: w_elec

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  LP_ELEC        1      0.112463    0.021989       5.11      0.0000
  LP_GAS         1     -0.097938    0.019300      -5.07      0.0000
  LINCOME        1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: w_gas

  Number of Observations                       60
  SSE                                      0.


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## Krok 4 — Nałożenie teorii ekonomicznej i jej test

Teoria popytu mówi, że nominalne efekty cenowo-dochodowe powinny spełniać **jednorodność stopnia zerowego**: przeskalowanie wszystkich cen i dochodu pozostawia realny popyt niezmieniony, więc współczynniki cen i dochodu w obrębie równania sumują się do zera. Nakładamy to na udział energii elektrycznej za pomocą instrukcji `RESTRICT`.

SYSLIN ponownie dopasowuje system przy tym ograniczeniu i raportuje test F **Restriction Results** dotyczący tego, czy restrykcja jest zgodna z danymi — bezpośredni, łączny test hipotezy jednorodności.

In [4]:
PROCEDURA syslin DANE=energy ON;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;

   /* Homogeneity on the electricity share equation:
      price and income coefficients sum to zero. */
   restrict lp_elec + lp_gas + lincome = 0;
WYKONAJ;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: w_elec

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: w_gas

  Number of Observations                       60
  SSE                                      0.


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Krok 5 — Przechwycenie estymat do raportowania elastyczności

Na potrzeby raportu końcowego utrwalamy zbieżne współczynniki za pomocą `OUTEST=`. Powstały zbiór danych niesie jeden wiersz na równanie z estymatami wyrazu wolnego i nachylenia oraz statystykami dopasowania, które zasilają obliczenia elastyczności ceny własnej i krzyżowej przedsiębiorstwa przy udziałach na poziomie średnich z próby. Drukujemy go dla zapisu.

In [5]:
PROCEDURA syslin DANE=energy ON outest=demand_est;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
WYKONAJ;

PROCEDURA DRUKUJ DANE=demand_est noobs;
   TYTUŁ "Estymaty współczynników systemu popytu SUR";
WYKONAJ;
TYTUŁ;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: w_elec

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: w_gas

  Number of Observations                       60
  SSE                                      0.


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## Interpretacja wyników

**Spójność znaków.** Estymaty SUR odtwarzają strukturę substytucji wbudowaną w dane. W równaniu udziału energii elektrycznej **współczynnik własnego logarytmu ceny jest dodatni** (`LP_ELEC` = 0.112), a **współczynnik ceny krzyżowej jest ujemny** (`LP_GAS` = -0.098); w równaniu udziału gazu wzorzec jest lustrzany (własny `LP_GAS` = 0.114, krzyżowy `LP_ELEC` = -0.075). Każdy efekt ceny własnej i krzyżowej jest statystycznie istotny (każde `Pr > |t|` poniżej 0.005), więc znaki substytucji są mocno zidentyfikowane, a nie są artefaktami zaszumionego dopasowania.

**SUR wykorzystuje korelację międzyrównaniową.** **Macierz kowariancji międzymodelowej** pokazuje ujemny element pozadiagonalny (-0.000068): gospodarstwa wymieniają wydatki na energię elektryczną na wydatki na gaz, dokładnie jak skonstruowano. Ponieważ ta kowariancja jest niezerowa, łączna estymacja SUR jest bardziej efektywna niż dopasowanie każdego równania osobno — błędy standardowe SUR w Kroku 2 są jednolicie nieco mniejsze niż błędy standardowe OLS równanie po równaniu w Kroku 3 (na przykład błąd standardowy `LP_ELEC` dla gazu spada z 0.0264 w OLS do 0.0255 w SUR), przy niezmienionych estymatach punktowych. To ciaśniejsze wnioskowanie jest korzyścią z modelowania systemu zamiast dwóch osobnych regresji.

**Teoria się broni.** Nałożenie **jednorodności stopnia zerowego** na udział energii elektrycznej (współczynniki cen i dochodu sumujące się do zera) za pomocą `RESTRICT` dało test F Restriction Results równy 2.14 przy `Pr > F` = 0.149. Restrykcja **nie jest odrzucona**, więc przewidywanie jednorodności z teorii popytu konsumenta jest zgodne z tą próbą — przedsiębiorstwo może z pewnością wnieść ograniczone, spójne z teorią estymaty do raportowania.

**Zastosowanie operacyjne.** Tabela `OUTEST=` utrwala jeden wiersz na równanie z estymatami wyrazu wolnego i nachylenia oraz statystykami dopasowania. Te współczynniki przekładają się bezpośrednio na elastyczności ceny własnej i krzyżowej oraz elastyczność dochodową (wydatkową) przy udziałach na poziomie średnich z próby (`W_ELEC` ~ 0.44, `W_GAS` ~ 0.23 z Kroku 1), dając przedsiębiorstwu obronną, spójną z teorią podstawę do prognozowania popytu i scenariuszy projektowania taryf.